In [1]:
import pandas as pd

books = pd.read_csv('books-data/books-cleaned-with-predictions.csv')

In [6]:
from transformers import pipeline

classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", return_all_scores=True, top_k=None)
classifier("I am so happy today!")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7354.00it/s]


[[{'label': 'joy', 'score': 0.9615581035614014},
  {'label': 'surprise', 'score': 0.02761468105018139},
  {'label': 'sadness', 'score': 0.005511835217475891},
  {'label': 'neutral', 'score': 0.0030506637413054705},
  {'label': 'anger', 'score': 0.0015047610504552722},
  {'label': 'fear', 'score': 0.00040151263237930834},
  {'label': 'disgust', 'score': 0.0003584022633731365}]]

In [7]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [8]:
sentences = books["description"][0].split(".")
classifier(sentences)

[[{'label': 'surprise', 'score': 0.7296019792556763},
  {'label': 'neutral', 'score': 0.14038611948490143},
  {'label': 'fear', 'score': 0.06816230714321136},
  {'label': 'joy', 'score': 0.04794260114431381},
  {'label': 'anger', 'score': 0.00915636494755745},
  {'label': 'disgust', 'score': 0.0026284789200872183},
  {'label': 'sadness', 'score': 0.002122164238244295}],
 [{'label': 'neutral', 'score': 0.4493708610534668},
  {'label': 'disgust', 'score': 0.2735908329486847},
  {'label': 'joy', 'score': 0.10908335447311401},
  {'label': 'sadness', 'score': 0.09362750500440598},
  {'label': 'anger', 'score': 0.04047827795147896},
  {'label': 'surprise', 'score': 0.026970183476805687},
  {'label': 'fear', 'score': 0.006879052612930536}],
 [{'label': 'neutral', 'score': 0.646215558052063},
  {'label': 'sadness', 'score': 0.24273371696472168},
  {'label': 'disgust', 'score': 0.04342271387577057},
  {'label': 'surprise', 'score': 0.028300553560256958},
  {'label': 'joy', 'score': 0.0142114954

In [5]:
sentences[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives'

In [ ]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

def calculate_max_emotion_scores(predictions):
    
    max_scores = {label: [] for label in emotion_labels}
    
    for pred in predictions:
        sorted_predictions = sorted(pred, key=lambda x: x['label'])
        for idx, label in enumerate(emotion_labels):
            max_scores[label].append(sorted_predictions[idx]['score'])
                
    return {label: np.max(scores) for label, scores in max_scores.items()}

In [13]:
from tqdm import tqdm

isbn = []
emotion_scores = {label: [] for label in emotion_labels}


for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [13:27<00:00,  6.44it/s]


In [14]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [15]:
books = pd.merge(books, emotions_df, on="isbn13", how="left")

In [16]:
books.to_csv('books-data/books-with-emotion-scores.csv', index=False)